In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.RandomResizedCrop(128, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

In [3]:
import torch
import torch.nn as nn

class MyAmazingCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.features = nn.Sequential(
            # in channels, out channels, kernel size
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
        )
        
        self.pool = nn.AdaptiveAvgPool2d((4, 4))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='cuda')

In [5]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

ROOT_DIR = "/content/masters/gmm/lab2/data"
TRAIN_DIR = ROOT_DIR + "/train"
VAL_DIR = ROOT_DIR + "/val"
TEST_DIR = ROOT_DIR + "/test"

train_dataset = ImageFolder(TRAIN_DIR, transform=transform)
test_dataset = ImageFolder(TEST_DIR, transform=transform)
val_dataset = ImageFolder(VAL_DIR, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

In [6]:
def train(model, optimizer, scheduler, criterion, epochs, save_path):
    train_losses = []
    train_accs = []
    val_losses = []
    val_accs = []

    best_val_loss = float('inf')
    epochs_no_improve = 0
    patience = 15

    for epoch in range(epochs):

        model.train()
        train_loss = 0
        correct = 0
        total = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)    
            
            predicted = torch.argmax(outputs, dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        train_loss /= len(train_loader.dataset)
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)

                predicted = torch.argmax(outputs, dim=1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)
        
        val_loss /= len(val_loader.dataset)
        val_acc = correct / total
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        scheduler.step(val_loss)

        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f} Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered!")
                break
            
    return train_losses, train_accs, val_losses, val_accs

In [7]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

def metrics(model):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)

            outputs = model(images)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro")
    recall = recall_score(all_labels, all_preds, average="macro")
    f1 = f1_score(all_labels, all_preds, average="macro")
    cm = confusion_matrix(all_labels, all_preds)
    
    print("Accuracy:", acc)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)

    print("Confusion Matrix:")
    print(cm)

In [8]:
import numpy as np

def threshold_analysis(model, thresholds, device):
    model.eval()

    # Collect all outputs and labels once
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)  # use sigmoid for BCE
            all_probs.append(probs.cpu())
            all_labels.append(torch.nn.functional.one_hot(labels, num_classes=probs.shape[1]).float())

    all_probs = torch.cat(all_probs, dim=0).numpy()    # shape [num_samples, num_classes]
    all_labels = torch.cat(all_labels, dim=0).numpy()  # shape [num_samples, num_classes]

    best_global_threshold = None
    best_global_f1 = -1

    print("Global threshold analysis:")
    # Global threshold loop: same threshold for all classes
    for t in thresholds:
        preds = (all_probs > t).astype(int)
        f1 = f1_score(all_labels, preds, average="macro")
        print(f"Threshold {t:.2f}: F1={f1:.4f}")
        if f1 > best_global_f1:
            best_global_f1 = f1
            best_global_threshold = t

    print(f"Best global threshold: {best_global_threshold:.2f} with F1={best_global_f1:.4f}\n")

    # Per-class thresholds
    best_class_thresholds = []
    print("Per-class threshold analysis:")
    for c in range(all_labels.shape[1]):
        best_t = None
        best_f1 = -1
        for t in thresholds:
            preds_c = (all_probs[:, c] > t).astype(int)
            f1 = f1_score(all_labels[:, c], preds_c)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
        best_class_thresholds.append(best_t)
        print(f"Class {c}: Best threshold {best_t:.2f} with F1={best_f1:.4f}")

    # Compute metrics with best global threshold
    final_preds = (all_probs > best_global_threshold).astype(int)
    acc = accuracy_score(all_labels.argmax(axis=1), final_preds.argmax(axis=1))
    precision = precision_score(all_labels.argmax(axis=1), final_preds.argmax(axis=1), average="macro")
    recall = recall_score(all_labels.argmax(axis=1), final_preds.argmax(axis=1), average="macro")
    f1 = f1_score(all_labels.argmax(axis=1), final_preds.argmax(axis=1), average="macro")
    cm = confusion_matrix(all_labels.argmax(axis=1), final_preds.argmax(axis=1))

    print("\nMetrics using best global threshold:")
    print("Accuracy:", acc)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)
    print("Confusion Matrix:")
    print(cm)

    return best_global_threshold, best_class_thresholds

In [19]:
import torch.optim as optim

# First experiment
model = MyAmazingCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)
epochs = 50

tl, ta, vl, va = train(model, optimizer, scheduler, criterion, epochs, ROOT_DIR + "/model-1.pth")

Epoch 1: Train Loss=1.0883, Train Acc=0.3971 Val Loss=1.0596, Val Acc=0.4133
Epoch 2: Train Loss=1.0342, Train Acc=0.4686 Val Loss=0.9963, Val Acc=0.4489
Epoch 3: Train Loss=0.9737, Train Acc=0.5171 Val Loss=0.9241, Val Acc=0.5067
Epoch 4: Train Loss=0.9212, Train Acc=0.5286 Val Loss=0.8786, Val Acc=0.5556
Epoch 5: Train Loss=0.8996, Train Acc=0.5724 Val Loss=0.8378, Val Acc=0.6000
Epoch 6: Train Loss=0.8660, Train Acc=0.5857 Val Loss=0.8561, Val Acc=0.6089
Epoch 7: Train Loss=0.8780, Train Acc=0.5781 Val Loss=0.8176, Val Acc=0.6133
Epoch 8: Train Loss=0.8534, Train Acc=0.5733 Val Loss=0.8075, Val Acc=0.6000
Epoch 9: Train Loss=0.8308, Train Acc=0.5905 Val Loss=0.7729, Val Acc=0.6133
Epoch 10: Train Loss=0.8166, Train Acc=0.6257 Val Loss=0.8011, Val Acc=0.5956
Epoch 11: Train Loss=0.8237, Train Acc=0.5962 Val Loss=0.8096, Val Acc=0.6178
Epoch 12: Train Loss=0.8045, Train Acc=0.6190 Val Loss=0.7503, Val Acc=0.6711
Epoch 13: Train Loss=0.7855, Train Acc=0.6162 Val Loss=0.8305, Val Acc=0.

In [20]:
metrics(model)

Accuracy: 0.6133333333333333
Precision: 0.6072788065843621
Recall: 0.6133333333333334
F1: 0.6086995416074089
Confusion Matrix:
[[61  8  6]
 [ 8 43 24]
 [12 29 34]]


In [25]:
!cp $ROOT_DIR/model-1.pth drive/MyDrive/masters/gmm/lab2/model-1.pth

!ls drive/MyDrive/masters/gmm/lab2

data  model-1.pth


In [11]:
import random
import torch.optim as optim

num_trials = 10
results = []

best_score = float('inf')   # for loss (lower is better)
best_config = None

epochs = 15

random.seed(42)

for i in range(num_trials):
    lr = 10 ** random.uniform(-5, -2)

    print(f"\n=== Trial {i+1} | lr={lr:.6f} ===")

    model = MyAmazingCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=5, factor=0.5
    )

    save_path = f"{ROOT_DIR}/model-trial-{i}.pth"

    tl, ta, vl, va = train(model, optimizer, scheduler, criterion, epochs, save_path)

    # --- choose metric ---
    trial_best_val_loss = min(vl)
    trial_best_val_acc = max(va)

    results.append({
        "lr": lr,
        "best_val_loss": trial_best_val_loss,
        "best_val_acc": trial_best_val_acc,
        "model_path": save_path
    })

    # --- selection criterion (LOSS) ---
    if trial_best_val_loss < best_score:
        best_score = trial_best_val_loss
        best_config = results[-1]

print("\n=== BEST CONFIG (by val loss) ===")
print(best_config)


=== Trial 1 | lr=0.000828 ===


Epoch 1: Train Loss=1.0939, Train Acc=0.3695 Val Loss=1.0510, Val Acc=0.4400
Epoch 2: Train Loss=1.0352, Train Acc=0.4705 Val Loss=1.0046, Val Acc=0.4844
Epoch 3: Train Loss=0.9723, Train Acc=0.5133 Val Loss=0.9475, Val Acc=0.5600
Epoch 4: Train Loss=0.9021, Train Acc=0.5667 Val Loss=0.8755, Val Acc=0.5600
Epoch 5: Train Loss=0.8664, Train Acc=0.5629 Val Loss=0.9004, Val Acc=0.4889
Epoch 6: Train Loss=0.8397, Train Acc=0.5895 Val Loss=0.8695, Val Acc=0.5867
Epoch 7: Train Loss=0.8485, Train Acc=0.5771 Val Loss=0.8356, Val Acc=0.5733
Epoch 8: Train Loss=0.8035, Train Acc=0.6086 Val Loss=0.8517, Val Acc=0.5333
Epoch 9: Train Loss=0.7912, Train Acc=0.6162 Val Loss=0.8460, Val Acc=0.5644
Epoch 10: Train Loss=0.7796, Train Acc=0.6390 Val Loss=0.8039, Val Acc=0.6133
Epoch 11: Train Loss=0.7564, Train Acc=0.6390 Val Loss=0.7965, Val Acc=0.6000
Epoch 12: Train Loss=0.7876, Train Acc=0.6305 Val Loss=0.9126, Val Acc=0.5200
Epoch 13: Train Loss=0.7742, Train Acc=0.6267 Val Loss=0.8494, Val Acc=0.

In [13]:
model.load_state_dict(torch.load(ROOT_DIR + "/model-trial-5.pth", map_location=device))
metrics(model)

Accuracy: 0.68
Precision: 0.7034911824515785
Recall: 0.68
F1: 0.6612895661975416
Confusion Matrix:
[[65  7  3]
 [10 60  5]
 [13 34 28]]


In [33]:
!cp $ROOT_DIR/model-2.pth drive/MyDrive/masters/gmm/lab2/model-2.pth

!ls drive/MyDrive/masters/gmm/lab2

data  model-1.pth  model-2.pth


In [14]:
thresholds = np.arange(0.1, 0.91, 0.1)
best_global, best_per_class = threshold_analysis(model, thresholds, device)

Global threshold analysis:
Threshold 0.10: F1=0.5458
Threshold 0.20: F1=0.5794
Threshold 0.30: F1=0.6076
Threshold 0.40: F1=0.6387
Threshold 0.50: F1=0.6629
Threshold 0.60: F1=0.6407
Threshold 0.70: F1=0.5986
Threshold 0.80: F1=0.3634
Threshold 0.90: F1=0.1533
Best global threshold: 0.50 with F1=0.6629

Per-class threshold analysis:
Class 0: Best threshold 0.50 with F1=0.7711
Class 1: Best threshold 0.50 with F1=0.6326
Class 2: Best threshold 0.50 with F1=0.5851

Metrics using best global threshold:
Accuracy: 0.5822222222222222
Precision: 0.5677655677655679
Recall: 0.5822222222222223
F1: 0.4852878634017391
Confusion Matrix:
[[64  9  2]
 [10 65  0]
 [17 56  2]]


In [37]:
import torch
import torch.nn as nn

class SimplerCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.features = nn.Sequential(
            # in channels, out channels, kernel size
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Linear(32, 3)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.classifier(x)
        return x

In [39]:
# Third experiment
model = SimplerCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)
epochs = 25

tl, ta, vl, va = train(model, optimizer, scheduler, criterion, epochs, ROOT_DIR + "/model-3.pth")

Epoch 1: Train Loss=1.1872, Train Acc=0.2724 Val Loss=1.1198, Val Acc=0.2578
Epoch 2: Train Loss=1.1806, Train Acc=0.2838 Val Loss=1.1390, Val Acc=0.2311
Epoch 3: Train Loss=1.1659, Train Acc=0.2895 Val Loss=1.1375, Val Acc=0.2933
Epoch 4: Train Loss=1.1628, Train Acc=0.2781 Val Loss=1.1397, Val Acc=0.2622
Epoch 5: Train Loss=1.1462, Train Acc=0.3190 Val Loss=1.1312, Val Acc=0.2533
Epoch 6: Train Loss=1.1426, Train Acc=0.2933 Val Loss=1.1188, Val Acc=0.2933
Epoch 7: Train Loss=1.1288, Train Acc=0.3229 Val Loss=1.1180, Val Acc=0.2844
Epoch 8: Train Loss=1.1310, Train Acc=0.3219 Val Loss=1.1083, Val Acc=0.2889
Epoch 9: Train Loss=1.1223, Train Acc=0.3486 Val Loss=1.1050, Val Acc=0.3156
Epoch 10: Train Loss=1.1107, Train Acc=0.3667 Val Loss=1.0969, Val Acc=0.3244
Epoch 11: Train Loss=1.1072, Train Acc=0.3533 Val Loss=1.0912, Val Acc=0.3600
Epoch 12: Train Loss=1.1059, Train Acc=0.3752 Val Loss=1.0858, Val Acc=0.3911
Epoch 13: Train Loss=1.0945, Train Acc=0.3695 Val Loss=1.0843, Val Acc=0.

In [40]:
metrics(model)

Accuracy: 0.44
Precision: 0.46417822728564423
Recall: 0.44
F1: 0.39937170121989846
Confusion Matrix:
[[61  2 12]
 [33 13 29]
 [42  8 25]]


In [41]:
!cp $ROOT_DIR/model-3.pth drive/MyDrive/masters/gmm/lab2/model-3.pth

!ls drive/MyDrive/masters/gmm/lab2

data  model-1.pth  model-2.pth	model-3.pth


In [42]:
print(train_dataset.classes)

['car', 'cat', 'dog']
